# Chapter 4: Representing Data and Engineering Features
## *Introduction to Machine Learning with Python* Müller & Guido

---
> **Ringkasan Bab:** "Garbage in, garbage out." Kualitas fitur menentukan kualitas model. Bab ini membahas cara merepresentasikan dan mentransformasi data  fitur kategorik, interaksi fitur, binning, polynomial features, dan feature selection  agar model bisa belajar lebih efektif.

## 4.1 Mengapa Feature Engineering Penting?

> *"Applied machine learning is basically feature engineering."*  Andrew Ng

Algoritma ML hanya bisa bekerja dengan data **numerik**. Representasi data yang tepat dapat:
- Membuat model sederhana menjadi sangat akurat
- Mengurangi kebutuhan data berlabel
- Mempercepat training

**Tipe data yang perlu ditangani khusus:**
- Fitur kategorik (nominal/ordinal)
- Fitur kontinu yang perlu dibinning
- Interaksi antar fitur
- Fitur yang tidak relevan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Dataset dummy untuk eksplorasi feature engineering
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'usia'      : np.random.randint(20, 60, n),
    'pendapatan': np.random.normal(5000, 2000, n).clip(1000),
    'kota'      : np.random.choice(['Jakarta', 'Bandung', 'Surabaya', 'Medan'], n),
    'pendidikan': np.random.choice(['SD', 'SMP', 'SMA', 'S1', 'S2'], n),
    'gender'    : np.random.choice(['Pria', 'Wanita'], n),
    'target'    : np.random.randint(0, 2, n)
})

print("Dataset:")
print(df.head(8))
print("\nTipe data:")
print(df.dtypes)

## 4.2 Fitur Kategorik

Algoritma ML tidak bisa langsung memproses string. Ada dua pendekatan:

### 4.2.1 One-Hot Encoding (Dummy Variables)

Untuk fitur **nominal** (tidak ada urutan)  setiap kategori menjadi kolom biner baru.

Contoh: `kota = ['Jakarta', 'Bandung', 'Surabaya']` → 3 kolom biner

**Masalah dummy variable trap:** Jika ada k kategori, cukup buat k-1 kolom (drop_first=True) untuk menghindari multikolinearitas.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# --- Cara 1: Pandas get_dummies ---
df_encoded_pd = pd.get_dummies(df[['kota', 'gender']], drop_first=False)
print("One-Hot Encoding dengan Pandas:")
print(df_encoded_pd.head())
print("Shape:", df_encoded_pd.shape)

In [ ]:
# --- Cara 2: Scikit-learn OneHotEncoder ---
ohe = OneHotEncoder(sparse=False, handle_unknown='ignore')
X_cat = df[['kota', 'gender']].values
X_ohe = ohe.fit_transform(X_cat)

print("Kategori yang di-encode:", ohe.categories_)
print("\nShape hasil OHE:", X_ohe.shape)
print("3 baris pertama:\n", X_ohe[:3])

### 4.2.2 Ordinal Encoding

Untuk fitur **ordinal** (ada urutan logis)  encode sebagai angka berurutan.

Contoh: `pendidikan: SD < SMP < SMA < S1 < S2` → `0, 1, 2, 3, 4`

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Tentukan urutan kategori secara eksplisit
urutan_pendidikan = [['SD', 'SMP', 'SMA', 'S1', 'S2']]
oe = OrdinalEncoder(categories=urutan_pendidikan)

df['pendidikan_encoded'] = oe.fit_transform(df[['pendidikan']]).astype(int)

print("Mapping pendidikan → angka:")
print(df[['pendidikan', 'pendidikan_encoded']].drop_duplicates().sort_values('pendidikan_encoded'))

## 4.3 Binning (Diskritisasi Fitur Kontinu)

**Binning** = membagi fitur kontinu menjadi beberapa **interval (bin)** diskrit.

**Kapan berguna:**
- Model linear tidak bisa menangkap hubungan non-linear → binning membantu
- Data memiliki distribusi tidak merata
- Ingin mengurangi noise pada fitur kontinu

**Dua tipe binning:**
- **Uniform:** lebar setiap bin sama
- **Quantile:** setiap bin berisi jumlah sampel yang sama

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

usia = df[['usia']]

# Uniform binning
kbd_uniform = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')
usia_bin_uniform = kbd_uniform.fit_transform(usia)

# Quantile binning
kbd_quantile = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
usia_bin_quantile = kbd_quantile.fit_transform(usia)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['usia'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Usia (Kontinu)')
axes[0].set_xlabel('Usia')

axes[1].hist(usia_bin_uniform, bins=5, color='salmon', edgecolor='white')
axes[1].set_title('Uniform Binning (5 bin)')
axes[1].set_xlabel('Bin')

axes[2].hist(usia_bin_quantile, bins=5, color='lightgreen', edgecolor='white')
axes[2].set_title('Quantile Binning (5 bin)')
axes[2].set_xlabel('Bin')

plt.suptitle('Perbandingan Binning Strategies', fontsize=13)
plt.tight_layout()
plt.show()

print("Edge setiap bin (uniform):", kbd_uniform.bin_edges_)

## 4.4 Polynomial Features & Interaksi Fitur

Model linear hanya bisa mempelajari hubungan linear $y = w_0 + w_1 x_1 + w_2 x_2$.

Dengan menambahkan **fitur polinomial**, model linear bisa mempelajari hubungan non-linear:

$$y = w_0 + w_1 x_1 + w_2 x_2 + w_3 x_1^2 + w_4 x_1 x_2 + w_5 x_2^2$$

**`PolynomialFeatures(degree=2)`** otomatis membuat:
- Fitur asli: $x_1, x_2$
- Pangkat dua: $x_1^2, x_2^2$
- Interaksi: $x_1 \cdot x_2$

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression

# Data non-linear
np.random.seed(0)
X = np.random.uniform(-3, 3, 100).reshape(-1, 1)
y = 0.5 * X.ravel()**2 + X.ravel() + np.random.randn(100)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

# Bandingkan: linear vs polynomial degree 2 dan 5
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
degrees = [1, 2, 5]

X_plot = np.linspace(-3, 3, 300).reshape(-1, 1)

for ax, degree in zip(axes, degrees):
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly  = poly.transform(X_test)
    X_plot_poly  = poly.transform(X_plot)

    lr = LinearRegression().fit(X_train_poly, y_train)

    train_score = lr.score(X_train_poly, y_train)
    test_score  = lr.score(X_test_poly,  y_test)

    ax.scatter(X_train, y_train, c='blue', alpha=0.5, s=20, label='Train')
    ax.scatter(X_test,  y_test,  c='red',  alpha=0.5, s=20, label='Test')
    ax.plot(X_plot, lr.predict(X_plot_poly), color='black', linewidth=2)
    ax.set_title(f'Degree={degree}\nTrain R²={train_score:.3f}, Test R²={test_score:.3f}')
    ax.set_xlabel('X')
    ax.legend(fontsize=8)

plt.suptitle('Polynomial Features: Underfitting vs. Overfitting', fontsize=13)
plt.tight_layout()
plt.show()

## 4.5 Fitur Non-Linear: Transformasi Matematika

Untuk data yang tidak berdistribusi normal atau memiliki skala sangat lebar, **transformasi matematika** bisa membantu.

| Transformasi | Rumus | Kegunaan |
|---|---|---|
| **Log** | $\log(x+1)$ | Data sangat skewed (pendapatan, harga) |
| **Square Root** | $\sqrt{x}$ | Data hitungan (count data) |
| **Reciprocal** | $1/x$ | Data dengan nilai besar |
| **Box-Cox** | Otomatis | Normalisasi distribusi |

In [ ]:
# Data pendapatan yang sangat skewed
np.random.seed(42)
pendapatan = np.random.exponential(scale=5000, size=500)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(pendapatan, bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Pendapatan (Distribusi Asli)\nRight-skewed')
axes[0].set_xlabel('Pendapatan')

axes[1].hist(np.sqrt(pendapatan), bins=50, color='salmon', edgecolor='white')
axes[1].set_title('Setelah √ (Square Root)')
axes[1].set_xlabel('√Pendapatan')

axes[2].hist(np.log1p(pendapatan), bins=50, color='lightgreen', edgecolor='white')
axes[2].set_title('Setelah log(1+x)')
axes[2].set_xlabel('log(Pendapatan+1)')

plt.suptitle('Transformasi pada Distribusi Skewed', fontsize=13)
plt.tight_layout()
plt.show()

## 4.6 Feature Selection

**Feature Selection** = memilih subset fitur yang paling informatif dan membuang yang tidak relevan.

**Manfaat:**
- Mengurangi overfitting
- Mempercepat training
- Model lebih mudah diinterpretasikan

**Tiga strategi utama:**

1. **Univariate Statistics** — uji statistik setiap fitur secara independen
2. **Model-based Selection** — gunakan koefisien/importance dari model
3. **Iterative Selection (RFE)** — tambah/hapus fitur secara iteratif

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectPercentile, SelectFromModel, RFE, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42
)

print(f"Jumlah fitur asli: {X_train.shape[1]}")

In [ ]:
# --- Cara 1: Univariate Feature Selection ---
selector_uni = SelectPercentile(score_func=f_classif, percentile=50)  # ambil 50% terbaik
X_train_uni  = selector_uni.fit_transform(X_train, y_train)
X_test_uni   = selector_uni.transform(X_test)

print(f"[Univariate] Fitur terpilih: {X_train_uni.shape[1]}")

# Evaluasi akurasi
from sklearn.linear_model import LogisticRegression
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train_uni)
X_te_s = scaler.transform(X_test_uni)
lr = LogisticRegression(max_iter=10000).fit(X_tr_s, y_train)
print(f"[Univariate] Akurasi test: {lr.score(X_te_s, y_test):.3f}")

In [ ]:
# --- Cara 2: Model-based Selection (dengan Random Forest) ---
selector_model = SelectFromModel(
    RandomForestClassifier(n_estimators=100, random_state=42),
    threshold='median'
)
X_train_model = selector_model.fit_transform(X_train, y_train)
X_test_model  = selector_model.transform(X_test)

print(f"[Model-based] Fitur terpilih: {X_train_model.shape[1]}")

X_tr_s2 = scaler.fit_transform(X_train_model)
X_te_s2 = scaler.transform(X_test_model)
lr2 = LogisticRegression(max_iter=10000).fit(X_tr_s2, y_train)
print(f"[Model-based] Akurasi test: {lr2.score(X_te_s2, y_test):.3f}")

In [ ]:
# Visualisasi: fitur mana yang dipilih?
mask_uni   = selector_uni.get_support()
mask_model = selector_model.get_support()

fig, axes = plt.subplots(2, 1, figsize=(12, 4))

axes[0].matshow(mask_uni.reshape(1, -1), cmap='gray_r', aspect='auto')
axes[0].set_xticks(range(len(cancer.feature_names)))
axes[0].set_xticklabels(cancer.feature_names, rotation=90, fontsize=7)
axes[0].set_yticks([])
axes[0].set_title('Univariate Selection (hitam = dipilih)')

axes[1].matshow(mask_model.reshape(1, -1), cmap='gray_r', aspect='auto')
axes[1].set_xticks(range(len(cancer.feature_names)))
axes[1].set_xticklabels(cancer.feature_names, rotation=90, fontsize=7)
axes[1].set_yticks([])
axes[1].set_title('Model-based Selection (hitam = dipilih)')

plt.tight_layout()
plt.show()

## 4.7 Handling Missing Values

Data dunia nyata hampir selalu punya nilai yang hilang (NaN). Strategi:

| Strategi | Cara | Cocok untuk |
|---|---|---|
| **Mean/Median/Mode imputation** | Isi dengan statistik kolom | Data MCAR (hilang acak) |
| **Constant imputation** | Isi dengan nilai tetap (0, 'unknown') | Ketika 'hilang' bermakna |
| **KNN imputation** | Isi berdasarkan tetangga terdekat | Data kompleks |
| **Drop rows/columns** | Hapus baris/kolom dengan NaN | Sedikit missing, data banyak |

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

# Buat data dengan missing values
np.random.seed(42)
X_missing = cancer.data.copy().astype(float)
# Masukkan 10% missing values secara acak
mask = np.random.rand(*X_missing.shape) < 0.1
X_missing[mask] = np.nan

print(f"Total missing values: {np.isnan(X_missing).sum()}")
print(f"Persentase missing  : {np.isnan(X_missing).mean():.1%}")

# Mean imputation
imputer_mean = SimpleImputer(strategy='mean')
X_imputed_mean = imputer_mean.fit_transform(X_missing)
print(f"\nSetelah Mean Imputation — missing values: {np.isnan(X_imputed_mean).sum()}")

# KNN imputation  
imputer_knn = KNNImputer(n_neighbors=5)
X_imputed_knn = imputer_knn.fit_transform(X_missing)
print(f"Setelah KNN Imputation — missing values : {np.isnan(X_imputed_knn).sum()}")

##  Ringkasan Bab 4

| Teknik | Kegunaan |
|---|---|
| **One-Hot Encoding** | Fitur kategorik nominal → vektor biner |
| **Ordinal Encoding** | Fitur kategorik ordinal → angka berurutan |
| **Binning** | Fitur kontinu → kategori diskrit |
| **Polynomial Features** | Tangkap hubungan non-linear dengan model linear |
| **Log/Sqrt Transform** | Normalkan distribusi yang skewed |
| **Univariate Selection** | Pilih fitur berdasarkan uji statistik |
| **Model-based Selection** | Pilih fitur berdasarkan importance model |
| **Imputation** | Tangani missing values |

---
 **Next:** Chapter 5 — Model Evaluation and Improvement